In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama

# -----------------------------------
# STEP 1: Load PDF
# -----------------------------------

loader = PyPDFLoader("company.pdf")

documents = loader.load()

print("PDF loaded successfully")


# -----------------------------------
# STEP 2: Split PDF into chunks
# -----------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))


# -----------------------------------
# STEP 3: Create FREE embeddings
# -----------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings created")


# -----------------------------------
# STEP 4: Store in Chroma
# -----------------------------------

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector database created")


# -----------------------------------
# STEP 5: Load local LLM
# -----------------------------------

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)


# -----------------------------------
# STEP 6: Ask question
# -----------------------------------

question = input("\nAsk your question: ")


# -----------------------------------
# STEP 7: Retrieve relevant documents
# -----------------------------------

results = vector_db.similarity_search(
    question,
    k=3
)

print("\nRelevant information found.")


# -----------------------------------
# STEP 8: Create context
# -----------------------------------

context = "\n\n".join(
    document.page_content
    for document in results
)


# -----------------------------------
# STEP 9: Create prompt
# -----------------------------------

prompt = f"""
You are a company policy assistant.

Answer the question using ONLY the information
provided in the context.

If the answer is not available in the context,
say "Information not available in the document."

Context:
{context}

Question:
{question}

Answer:
"""


# -----------------------------------
# STEP 10: Generate answer
# -----------------------------------

response = llm.invoke(prompt)


# -----------------------------------
# STEP 11: Display answer
# -----------------------------------

print("\n================================")
print("ANSWER")
print("================================")

print(response.content)

PDF loaded successfully
Number of chunks: 5


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings created
Vector database created

Ask your question: working hours?

Relevant information found.

ANSWER
The standard working hours are 9:00 AM to 6:00 PM, Monday to Friday, with a one-hour lunch break.


In [ ]:
# Install necessary libraries
!pip install langchain_community pypdf langchain-text-splitters langchain-huggingface chromadb langchain-ollama

In [ ]:
import subprocess
import time

# Install zstd, a utility required by Ollama installer
!apt-get update
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in the background
# Start Ollama server in a new process
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give Ollama a moment to start up
time.sleep(5)
print("Ollama server started.")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [ ]:
# Pull the llama3.2 model
!/usr/local/bin/ollama pull llama3.2
print("llama3.2 model pulled.")


llama3.2 model pulled.
